In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [10]:
import heapq
import math

from typing import List

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + F.tanh(torch.sqrt(torch.FloatTensor([2/math.pi]))[0] * (x + 0.044715 * (x ** 3))))
    

class TokenEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, emb_size: int):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, emb_size)

    def forward(self, x: torch.Tensor):
        return self.embeddings(x)

class PositionalEmbeddings(nn.Module):
    def __init__(self, max_seq_len:  int, emb_size: int):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.emb_size = emb_size

        self.embeddings = nn.Embedding(max_seq_len, emb_size)

    def forward(self, seq_len: int, start_pos: int = 0):
        return self.embeddings.weight[start_pos: start_pos + seq_len]

import math 

class HeadAttention(nn.Module):
    def __init__(self, emb_size: int, head_size:int, max_seq_len: int):
        super().__init__()
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len

        self.W_k = nn.Linear(self.emb_size, self.head_size)
        self.W_q = nn.Linear(self.emb_size, self.head_size)
        self.W_v = nn.Linear(self.emb_size, self.head_size)

        self.mask = torch.tril(torch.ones(self.max_seq_len, self.max_seq_len))

    def forward(self, x: torch.Tensor,
                 use_cache: bool = True, cache: tuple = None):
        batch_size, seq_len, emb_size = x.shape
        key = self.W_k(x)
        if cache is not None:
            key = torch.cat([cache[0], key], 1)
        query = self.W_q(x)
        value = self.W_v(x)
        if cache is not None:
            value = torch.cat([cache[1], value], 1)

        attention_matrix : torch.Tensor = query @ key.transpose(1,2) / math.sqrt(self.head_size)

        if cache is None:
            mask = self.mask[:seq_len, :seq_len] == 0
            attention_matrix = attention_matrix.masked_fill(mask, float('-inf'))
        
        attention_matrix = torch.softmax(attention_matrix, 2) 

        x = attention_matrix @ value
        
        if not use_cache:  
            return (x, None)

        if use_cache:
            return (x, (key, value))


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads: int, emb_size: int,
                head_size: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.dropout = dropout

        self.head_attentions = nn.ModuleList([HeadAttention(self.emb_size, self.head_size, self.max_seq_len) for i in range(self.num_heads)])
        self.otpt = nn.Linear(self.head_size * self.num_heads, self.emb_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, use_cache: bool=True, cache=None):
        if cache is not None:    
            tensors_and_caches = [head_attention(x, use_cache, ch) for ch, head_attention in zip(cache, self.head_attentions)]
        else:
            tensors_and_caches = [head_attention(x, use_cache, cache) for head_attention in self.head_attentions]
        tensors = [ tensor for tensor, _ in tensors_and_caches]
        cache_new = [cache for _, cache in tensors_and_caches]
            
        tensor = torch.cat(tensors, dim=2)
        tensor = self.otpt(tensor)
        tensor = self.dropout(tensor)

        if use_cache:
            return (tensor, cache_new)

        return (tensor, None)
        
class FeedForward(nn.Module):
    def __init__(self, emb_size: int, dropout: float == 0.1):
        super().__init__()
        self.emb_size = emb_size
        self.dropout = dropout

        self.first_linear = nn.Linear(emb_size, emb_size * 4)
        self.gelu = GELU()
        self.second_linear = nn.Linear(4 * emb_size, emb_size)
        self.drpt = nn.Dropout(self.dropout)

    def forward(self, x: torch.Tensor):
        tensor = self.first_linear(x)
        tensor = self.gelu(tensor)
        tensor = self.second_linear(tensor)
        tensor = self.drpt(tensor)
        return tensor

class Decoder(nn.Module):
    def __init__(self, num_heads: int, emb_size: int,
                head_size: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.dropout = dropout

        self.multi_head_attention = MultiHeadAttention(self.num_heads, self.emb_size, self.head_size, self.max_seq_len, self.dropout)
        self.feed_forward = FeedForward(self.emb_size, self.dropout)
        self.first_norm = nn.LayerNorm(self.emb_size)
        self.second_norm = nn.LayerNorm(self.emb_size)

    def forward(self, x: torch.Tensor, use_cache = True, cache = None):
        tensor = self.first_norm(x)
        tensor, cache = self.multi_head_attention(tensor, use_cache, cache)
        tensor = x + tensor
        tensor = tensor + self.feed_forward(self.second_norm(tensor))
        if use_cache:
            return (tensor, cache)

        return (tensor, None)


class GPT2(nn.Module):
    def __init__(self, vocab_size: int, max_seq_len: int, emb_size: int,
                num_heads: int, head_size: int, num_layers: int,
                dropout: float = 0.1, device: str = "cpu"):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.head_size = head_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.device = device

        self.token_embeddings = TokenEmbeddings(self.vocab_size, self.emb_size)
        self.positional_embeddings = PositionalEmbeddings(self.max_seq_len, self.emb_size)
        self.drpt = nn.Dropout(self.dropout)
        self.decoders = [Decoder(self.num_heads, self.emb_size, self.head_size, self.max_seq_len, self.dropout) for i in range(self.num_layers)]
        self.linear = nn.Linear(self.emb_size, self.vocab_size)
        self.norm = nn.LayerNorm(self.emb_size)

    def forward(self, x: torch.Tensor, use_cache=True, cache=None):
        if cache is None:
            positional_embeddings = self.positional_embeddings(x.shape[1])
        else:
            key, value = cache[0][0]
            start_pos = key.shape[1]
            positional_embeddings = self.positional_embeddings(1, start_pos)
        tensor = self.token_embeddings(x) + positional_embeddings
        tensor = self.drpt(tensor)
        new_caches = []
        if cache is not None:
            for ch, decoder in zip(cache,self.decoders):
                tensor, cache_decoder = decoder(tensor, use_cache, ch)
                new_caches.append(cache_decoder)
        else:
            for decoder in self.decoders:
                tensor, cache_decoder = decoder(tensor, use_cache, cache)
                new_caches.append(cache_decoder)
        tensor = self.norm(tensor)
        tensor = self.linear(tensor)
        if use_cache:
            return (tensor, new_caches)
        
        return (tensor, None) 
        
        
    def generate(self, x: torch.Tensor, max_new_tokens: int, do_sample: bool, temperature: float = 1.0,
                top_k: int = None, top_p: float = None, use_cache=True):
        cache = None
        for i in range(max_new_tokens):
            if cache is None:
                x_hat = x[:, :]
            else:
                x_hat = x[:, -1:]
            
            logits, cache = self.forward(x_hat, use_cache=use_cache, cache=cache)
            logits = logits[:, -1, :]
            logits /= temperature
            if not do_sample:
                probas = logits.softmax(1)
                new_tokens = torch.argmax(probas, dim=1)
            else:
                if top_k is not None:
                    new_logits = []
                    for j in range(logits.shape[0]):
                        logits_j = logits[j,:]
                        topk_values, topk_indicies = torch.topk(logits_j, top_k)
                        mask = torch.ones_like(logits_j) == 1
                        mask[topk_indicies] = 0
                        logits_j.masked_fill_(mask, float('-inf'))
                        new_logits.append(logits_j)
                    logits = torch.stack(new_logits,)
                if top_p is not None:
                    new_logits = []
                    for j in range(logits.shape[0]):
                        logits_j = logits[j,:]
                        probabilities = logits_j.softmax(0)
                        
                        max_probabilities = sorted(probabilities.detach().numpy(),reverse=True)

                        cum_proba = max_probabilities[0]
                        min_proba = max_probabilities[0]
                        m = 1
                        while m < len(max_probabilities):
                            cum_proba += max_probabilities[m]
                            if cum_proba < top_p:
                                min_proba = max_probabilities[m]
                            else:
                                break
                            m += 1
                        mask = probabilities  < min_proba

                        logits_j.masked_fill_(mask, float('-inf'))
                        new_logits.append(logits_j)
                    logits = torch.stack(new_logits,)
                        
                probas = logits.softmax(1)
                new_tokens = torch.multinomial(probas, 1)
            new_tokens = torch.reshape(new_tokens, (new_tokens.shape[0], 1))
            x = torch.cat([x, new_tokens], dim=1)
        return x

    def save(self, path):
        torch.save({
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_seq_len': self.max_seq_len,
            'emb_size': self.emb_size,
            'num_heads': self.num_heads,
            'head_size': self.head_size,
            'num_layers': self.num_layers
        }, path)
        
    @classmethod
    def load(cls, path, device):
        checkpoint = torch.load(path, map_location=device)
        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_seq_len=checkpoint['max_seq_len'],
            emb_size=checkpoint['emb_size'],
            num_heads=checkpoint['num_heads'],
            head_size=checkpoint['head_size'],
            num_layers=checkpoint['num_layers']
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        return model
    
    def fit(self, train_loader , valid_loader, 
           num_epoch: int, learning_rate: float,
           use_cache=False):

        self.to(self.device)

        criterion = nn.CrossEntropyLoss()

        adam = torch.optim.Adam(self.parameters(), lr=learning_rate)

        for epoch in range(num_epoch):
            self.train()

            for inputs, targets in train_loader:
                batch_size, seq_len = inputs.shape
                outputs, cache = self.forward(inputs, use_cache=use_cache)

                outputs = outputs.reshape(batch_size * seq_len, self.vocab_size)
                targets = targets.reshape(batch_size * seq_len)

                self.loss = criterion(outputs, targets)

                adam.zero_grad()
                self.loss.backward()

                adam.step()

            self.eval()
    
            with torch.no_grad():
                for inputs, targets in valid_loader:
                    
                    batch_size, seq_len = inputs.shape
                    outputs, cache = self.forward(inputs, use_cache=use_cache)
    
                    outputs = outputs.reshape(batch_size * seq_len, self.vocab_size)
                    targets = targets.reshape(batch_size * seq_len)
    
                    self.loss = criterion(outputs, targets)

